DSCI 552 Homework 4

Name: Brynn Dafoe GitHub Username: brynndafoe02 USD ID: 3109-6692-10

HOMEWORK 3 PORTION NEEDED FOR HOMEWORK 4:

In [57]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

from itertools import combinations
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path
import re

In [58]:
path_to_AReM = Path("../data/AReM")
column_names = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]

training_dfs = {}
testing_dfs = {}
# training and testing dataframes that match the structure of the original files

for data_dir in path_to_AReM.iterdir():
    if data_dir.is_dir():
        dir_name = data_dir.name
            
        for file in data_dir.iterdir():
            file_name = Path(file).stem
            file_num = int("".join(re.findall(r"\d", file_name)))

            key = f"{dir_name}_{file_name}"
            
            if (dir_name == "bending1") or (dir_name == "bending2"):
                df = pd.read_csv(file, skiprows=5, names=column_names, sep = None, engine = "python")
                # if dir_name == "bending1":
                #     df = pd.read_csv(file, skiprows=5, names=column_names)
                # else:
                #     df = pd.read_csv(file, skiprows=5, names=column_names, sep=" ")
                
                if (file_num == 1) or (file_num == 2):
                    testing_dfs[key] = df
                else:
                    training_dfs[key] = df
            else: 
                df = pd.read_csv(file, skiprows=5, names=column_names, sep = None, engine = "python")
                
                if (file_num == 1) or (file_num == 2) or (file_num == 3):
                    testing_dfs[key] = df
                else:
                    training_dfs[key] = df

In [59]:
columns_six = column_names[1:]

training_features = []
# 69 rows x 42 columns
# looks like:
    # [ {dirfilename : name, min1 : x, max1 : x, mean1 : x, median1 : x, stddev : x,
    # firstQ1 : x, thirdQ1 : x, ........, thirdQ6 : x}, {}, {}, ..., {} ]
    # for each of the 6 features in files: 
        # avg_rss12, var_rss12, avg_rss13, var_rss13, vg_rss23, ar_rss23

for dirfile1, its_df1 in training_dfs.items():
    a_instance = {}
    a_instance["dirfile_name"] = dirfile1 # to keep track of where the numbers are coming from

    for i, a_column in enumerate(columns_six, start=1):
        a_instance[f"min_{i}"] = its_df1[a_column].min()
        a_instance[f"max_{i}"] = its_df1[a_column].max()
        a_instance[f"mean_{i}"] = its_df1[a_column].mean()
        a_instance[f"median_{i}"] = its_df1[a_column].median()
        a_instance[f"std_dev_{i}"] = its_df1[a_column].std()
        a_instance[f"first_quart_{i}"] = its_df1[a_column].quantile(0.25)
        a_instance[f"third_quart_{i}"] = its_df1[a_column].quantile(0.75)

    training_features.append(a_instance)

train_feat_df = pd.DataFrame(training_features)


In [60]:
testing_features = []
# 19 rows x 42 columns

for dirfile2, its_df2 in testing_dfs.items():
    b_instance = {}
    b_instance["dirfile_name"] = dirfile2

    for j, b_column in enumerate(columns_six, start=1):
        b_instance[f"min_{j}"] = its_df2[b_column].min()
        b_instance[f"max_{j}"] = its_df2[b_column].max()
        b_instance[f"mean_{j}"] = its_df2[b_column].mean()
        b_instance[f"median_{j}"] = its_df2[b_column].median()
        b_instance[f"std_dev_{j}"] = its_df2[b_column].std()
        b_instance[f"first_quart_{j}"] = its_df2[b_column].quantile(0.25)
        b_instance[f"third_quart_{j}"] = its_df2[b_column].quantile(0.75)

    testing_features.append(b_instance)

test_feat_df = pd.DataFrame(testing_features)


In [61]:
all_features = training_features + testing_features
all_features_df = pd.DataFrame(all_features)
af_df = all_features_df.drop(columns=["dirfile_name"])
# ALL time-domain features for each original feature
std_of_features = af_df.std()

In [62]:
boostrap_results = []

for a_column in af_df.columns:
    one_feat = {}
    one_feat["feat"] = a_column
    one_feat["std_dev"] = std_of_features[a_column]

    boostrap_samples = []

    for _ in range(1000):
        boostrap_sample = af_df[a_column].sample(frac=1.0, replace=True)
        boostrap_sample_std = boostrap_sample.std()
        boostrap_samples.append(boostrap_sample_std)

    bootstrap_series = pd.Series(boostrap_samples)

    left_tail = bootstrap_series.quantile(0.05)
    right_tail = bootstrap_series.quantile(0.95)

    one_feat["left_tail"] = left_tail
    one_feat["right_tail"] = right_tail

    boostrap_results.append(one_feat)


# for result in boostrap_results:
#     for key, value in result.items():
#         print(f"{key} -> {value}")
#     print("\n")

In HW3, I chose Max, Median, and Third Quartile to be the three features.

HOMEWORK 4 SECTION:

Question 2.a.i
- Assume that you want to use the training set to classify bending from other activities, ie: you have a binary classification problem
- Depict scatter plots of the features you specified in 1.c.iv extracted from time series 1, 2, and 6 of each instance, and use color to distinguish bending vs. other activities
- (See p. 129 of TB)
- (Some LogReg packages have built in L2 regularization -> to remove the effect of L2 regularization, set lambda = 0 or set the budget C -> infinity (ie: very large value))
- (Can repeat this experiment with other features as well as with time series 3, 4, and 5 in each instance)

In [63]:
# max_1, max_2, max_6
# median_1, median_2, median_6
# third_quart_1, third_quart_2, third_quart_6

# train_feat_df -> {dirfilename : name, min1 : x, etc...} -> 69 rows
    # bending1, bending1 -> bending
    # cycling, lying, sitting, standing, walking -> 1, 2, 6 -> other

# scatterplot: bending = red, other = blue
            #  x-axis = 1 feature (like max_1)
            #  y-axis = 1 feature (like max_2)
# maxes: max_1 + max_2, max_1 + max_6, max_2 + max_6
# medians: median_1 + median_2, median_1 + median_6, median_2 + median_6
# third_quarts: third_quart_1 + third_quart_2, third_quart_1 + third_quart_6, third_quart_2 + third_quart_6

max_1s = {"bending" : [], "other" : []}
max_2s = {"bending" : [], "other" : []}
max_6s = {"bending" : [], "other" : []}

median_1s = {"bending" : [], "other" : []}
median_2s = {"bending" : [], "other" : []}
median_6s = {"bending" : [], "other" : []}

thirdQ_1s = {"bending" : [], "other" : []}
thirdQ_2s = {"bending" : [], "other" : []}
thirdQ_6s = {"bending" : [], "other" : []}

for _, row in train_feat_df.iterrows(): # _ because we don't care about index, just go through entire thing
    
    file_name = row["dirfile_name"]
    print(file_name)
    
    if "bend" in file_name:
        activity = "bending"
    else:
        activity = "other"
        
    max_1s[activity].append(row["max_1"])
    print(f"max_1: {row["max_1"]}")
    max_2s[activity].append(row["max_2"])
    print(f"max_2: {row["max_2"]}")
    max_6s[activity].append(row["max_6"])
    print(f"max_6: {row["max_6"]}")

    median_1s[activity].append(row["median_1"])
    median_2s[activity].append(row["median_2"])
    median_6s[activity].append(row["median_6"])

    thirdQ_1s[activity].append(row["third_quart_1"])
    thirdQ_2s[activity].append(row["third_quart_2"])
    thirdQ_6s[activity].append(row["third_quart_6"])
    print()

for key, value in max_1s.items():
    print(f"{key} -> {value}")


bending1_dataset7
max_1: 48.0
max_2: 1.5
max_6: 2.96

bending1_dataset6
max_1: 48.0
max_2: 1.58
max_6: 5.26

bending1_dataset4
max_1: 47.75
max_2: 3.0
max_6: 2.18

bending1_dataset5
max_1: 45.75
max_2: 2.83
max_6: 1.79

bending1_dataset3
max_1: 47.4
max_2: 1.7
max_6: 1.79

walking_dataset7
max_1: 47.67
max_2: 12.48
max_6: 8.01

walking_dataset6
max_1: 51.0
max_2: 12.21
max_6: 10.21

walking_dataset4
max_1: 46.0
max_2: 16.2
max_6: 8.5

walking_dataset5
max_1: 46.25
max_2: 12.68
max_6: 9.39

walking_dataset10
max_1: 51.25
max_2: 13.55
max_6: 8.32

walking_dataset11
max_1: 45.33
max_2: 14.67
max_6: 8.32

walking_dataset13
max_1: 46.0
max_2: 12.47
max_6: 10.0

walking_dataset12
max_1: 45.5
max_2: 13.47
max_6: 9.67

walking_dataset15
max_1: 44.0
max_2: 13.86
max_6: 9.0

walking_dataset14
max_1: 46.25
max_2: 14.82
max_6: 9.51

walking_dataset8
max_1: 45.75
max_2: 15.37
max_6: 8.86

walking_dataset9
max_1: 43.67
max_2: 17.24
max_6: 9.42

bending2_dataset6
max_1: 47.5
max_2: 6.38
max_6: 4.92

